In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType

In [0]:
df_taxas_silver = spark.table("projeto_seguranca_rj.bronze.taxas_municipio")

In [0]:
df_taxas_silver.printSchema()

In [0]:
df_taxas_silver = (
    df_taxas_silver
    .withColumn(
        "fmun_cod",
        F.col("fmun_cod").cast("string")
    )
)

In [0]:
colunas_taxas = [
    "hom_doloso",
    "lesao_corp_morte",
    "latrocinio",
    "cvli",
    "hom_por_interv_policial",
    "letalidade_violenta",
    "tentat_hom",
    "lesao_corp_dolosa",
    "estupro",
    "hom_culposo",
    "lesao_corp_culposa",
    "roubo_transeunte",
    "roubo_celular",
    "roubo_em_coletivo",
    "roubo_rua",
    "roubo_veiculo",
    "roubo_carga",
    "roubo_comercio",
    "roubo_residencia",
    "roubo_banco",
    "roubo_cx_eletronico",
    "roubo_conducao_saque",
    "roubo_apos_saque",
    "roubo_bicicleta",
    "outros_roubos",
    "total_roubos",
    "furto_veiculos",
    "furto_transeunte",
    "furto_coletivo",
    "furto_celular",
    "furto_bicicleta",
    "outros_furtos",
    "total_furtos",
    "sequestro",
    "extorsao",
    "sequestro_relampago",
    "estelionato",
    "apreensao_drogas",
    "posse_drogas",
    "trafico_drogas",
    "apreensao_drogas_sem_autor",
    "recuperacao_veiculos",
    "apf",
    "aaapai",
    "cmp",
    "cmba",
    "ameaca",
    "pessoas_desaparecidas",
    "encontro_cadaver",
    "encontro_ossada",
    "pol_militares_mortos_serv",
    "pol_civis_mortos_serv",
    "registro_ocorrencias"
]

for coluna in colunas_taxas:
    df_taxas_silver = (
        df_taxas_silver
        .withColumn(
            coluna,
            F.regexp_replace(
                F.col(coluna),
                ",",
                "."
            ).cast("double")
        )
    )

In [0]:
df_taxas_silver.printSchema()

In [0]:
nulos_taxas = (
    df_taxas_silver
    .select(
        [
            F.count(
                F.when(
                    F.col(c).isNull(),
                    c
                )
            ).alias(c)
            for c in colunas_taxas
        ]
    )
    .collect()[0]
    .asDict()
)

problemas_taxas = {
    coluna: quantidade
    for coluna, quantidade in nulos_taxas.items()
    if quantidade > 0
}

if problemas_taxas:
    raise ValueError(
        "Foram encontrados valores inválidos "
        f"após a conversão das taxas: {problemas_taxas}"
    )

print(
    "Conversão das 53 colunas de taxas "
    "validada com sucesso."
)

In [0]:
nulos_chave = (
    df_taxas_silver
    .filter(
        F.col("fmun_cod").isNull()
        | F.col("fmun").isNull()
        | F.col("ano").isNull()
        | F.col("mes").isNull()
    )
    .count()
)

if nulos_chave > 0:
    raise ValueError(
        f"Foram encontrados {nulos_chave} registros "
        "com valores nulos nos campos obrigatórios."
    )

print("Campos obrigatórios validados com sucesso.")

In [0]:
df_taxas_silver = (
    df_taxas_silver
    .withColumn(
        "regiao",
        F.coalesce(
            F.col("regiao"),
            F.lit("NAO_INFORMADO")
        )
    )
)

In [0]:
nulos_regiao = (
    df_taxas_silver
    .filter(
        F.col("regiao").isNull()
    )
    .count()
)

if nulos_regiao > 0:
    raise ValueError(
        f"Ainda existem {nulos_regiao} "
        "valores nulos em regiao."
    )

print(
    "Tratamento dos valores nulos "
    "de regiao validado com sucesso."
)

In [0]:
df_taxas_silver.groupBy("regiao").count().show()

In [0]:
duplicados_chave = (
    df_taxas_silver
    .groupBy(
        "fmun_cod",
        "ano",
        "mes"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

qtd_duplicados = duplicados_chave.count()

print(
    "Chaves duplicadas:",
    qtd_duplicados
)

if qtd_duplicados > 0:
    display(duplicados_chave)

    raise ValueError(
        "Foram encontradas chaves duplicadas "
        "em taxas_municipio."
    )

print("Unicidade da chave validada com sucesso.")

In [0]:
(
    df_taxas_silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "projeto_seguranca_rj.silver.taxas_municipio"
    )
)

print(
    "Tabela projeto_seguranca_rj.silver.taxas_municipio "
    "salva com sucesso."
)

In [0]:
linhas_dataframe = df_taxas_silver.count()

linhas_tabela = (
    spark.table(
        "projeto_seguranca_rj.silver.taxas_municipio"
    )
    .count()
)

print(
    f"Silver taxas: DataFrame={linhas_dataframe} | "
    f"Tabela={linhas_tabela}"
)

if linhas_dataframe != linhas_tabela:
    raise ValueError(
        "Divergência entre o DataFrame tratado "
        "e a tabela Silver persistida."
    )

print(
    "Persistência da Silver taxas "
    "validada com sucesso."
)